# How much of a protein can you change before it stops being that protein?

**Protein design practical — University of Tartu**

You will take a real protein, delete part of its sequence, and ask a generative
model to invent replacements. Then you will measure what survived: how much of
the original sequence came back, whether the amino-acid composition still looks
like a protein, and whether the fold held.

Along the way you will catch the model doing something it should not.

---

## ⚠️ Do this first

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. **Run the next cell now.** It installs the model and downloads ~5 GB of
   weights, which takes several minutes. Start it and leave it running while
   we talk.

If no GPU is available the notebook will fall back to CPU automatically — it
still works, just more slowly, so stick to the shorter proteins.


In [ ]:
#@title ⚙️ Install ESM3 and load the model  (several minutes)
%%time
import subprocess, sys, os

print('installing esm ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'esm', 'py3Dmol'],
               check=True)

import torch
from esm.models.esm3 import ESM3
from esm.sdk.api import ESMProtein, GenerationConfig

# T4 GPUs have no bfloat16; float32 is the safe choice everywhere.
if torch.cuda.is_available():
    DEVICE = 'cuda'
    print('GPU:', torch.cuda.get_device_name(0))
else:
    DEVICE = 'cpu'
    print('No GPU - running on CPU. Use the shorter proteins.')

print('downloading model weights (~5 GB, once) ...')
model = ESM3.from_pretrained('esm3-sm-open-v1').to(DEVICE).float()

# smoke test - fail now rather than three cells later
_p = model.generate(ESMProtein(sequence='_' * 12),
                    GenerationConfig(track='sequence', num_steps=12, temperature=0.7))
assert isinstance(_p.sequence, str) and len(_p.sequence) == 12, 'smoke test failed'
print(f'\n✅ ready on {DEVICE}  (test output: {_p.sequence})')


---
## The experiment

Pick a protein and a **masking fraction**. That fraction of its residues is
replaced with blanks, and ESM3 fills them back in — it sees the rest of the
sequence but not the parts you removed.

Three numbers come back:

| Number | What it tells you |
|---|---|
| **overall identity** | how much of the whole sequence matches the original |
| **identity at masked positions** | whether it *recovered* the original residues or *invented* new ones |
| **top residue %** | sanity check. A real protein's commonest residue is 8–12%. Much above that and the model has collapsed into gibberish |

Start around 30% and work outwards.


In [ ]:
#@title 🧬 Mask and regenerate  { display-mode: "form" }
protein = "Ubiquitin (76 aa)" #@param ["Trp-cage (20 aa, tiny)", "Villin headpiece (36 aa)", "Protein G B1 domain (56 aa)", "Ubiquitin (76 aa)", "GFP (238 aa, slow on CPU)", "paste my own below"]
my_own_sequence = "" #@param {type:"string"}
mask_fraction = 0.3 #@param {type:"slider", min:0.05, max:0.9, step:0.05}
temperature = 0.7 #@param ["0.3", "0.7", "1.0", "1.5"] {type:"raw"}
seed = 0 #@param {type:"integer"}

import random, collections, time, torch
from esm.sdk.api import ESMProtein, GenerationConfig

LIBRARY = {
        "Trp-cage (20 aa, tiny)": "NLYIQWLKDGGPSSGRPPPS",
        "Villin headpiece (36 aa)": "MLSDEDFKAVFGMTRSAFANLPLWKQQNLKKEKGLF",
        "Protein G B1 domain (56 aa)": "MTYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
        "Ubiquitin (76 aa)": "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG",
        "GFP (238 aa, slow on CPU)": "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"
    }

native = (my_own_sequence.strip().upper() if protein.startswith('paste')
          else LIBRARY[protein])
assert native, 'Paste a sequence, or pick one from the list.'
assert set(native) <= set('ACDEFGHIKLMNPQRSTVWY'), 'Sequence has non-standard letters.'

random.seed(seed)
idx = sorted(random.sample(range(len(native)), int(len(native) * mask_fraction)))
masked = ''.join('_' if i in set(idx) else a for i, a in enumerate(native))

t0 = time.time()
p = ESMProtein(sequence=masked)
p = model.generate(p, GenerationConfig(track='sequence',
                                       num_steps=max(len(idx), 1),
                                       temperature=temperature))
p = model.generate(p, GenerationConfig(track='structure', num_steps=8))
elapsed = time.time() - t0

new = p.sequence
overall = sum(a == b for a, b in zip(native, new)) / len(native)
at_masked = sum(native[i] == new[i] for i in idx) / max(len(idx), 1)
aa, n = collections.Counter(new).most_common(1)[0]
top = n / len(new)

p.to_pdb('design.pdb')

print(f'{protein}   {len(native)} residues, {len(idx)} masked ({mask_fraction:.0%})')
print(f'took {elapsed:.1f}s on {DEVICE}\n')
print(f'  overall identity            {overall:6.1%}')
print(f'  identity at masked positions{at_masked:6.1%}')
print(f'  most common residue         {aa} at {top:.1%}', end='  ')
print('← looks like a protein' if top < 0.20 else '← COLLAPSED, this is not a real protein')
print(f'\noriginal : {native}')
print(f'designed : {new}')
print('changed  : ' + ''.join(' ' if a == b else '^' for a, b in zip(native, new)))


---
## Did the fold survive?

ESM3 also predicts a structure for the sequence it wrote. Drag to rotate.
Compare what you see with the real fold — a β-barrel for GFP, a β-grasp for
ubiquitin, a three-helix bundle for villin.


In [ ]:
#@title 🔍 View the designed structure
import subprocess, sys
try:
    import py3Dmol
except ModuleNotFoundError:          # fresh runtime, or cell 1 not run
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'py3Dmol'], check=True)
    import py3Dmol
view = py3Dmol.view(width=720, height=520)
view.addModel(open('design.pdb').read(), 'pdb')
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.setBackgroundColor('white')
view.zoomTo()
view.show()


---
## Find the breaking point

One run tells you little. This sweeps the masking fraction and plots what
happens, so you can see *where* the protein stops being itself.

Watch the two lines diverge, and watch the composition line rise — that rise is
the model giving up and padding with one repeated residue.


In [ ]:
#@title 📈 Sweep the masking fraction
sweep_protein = "Ubiquitin (76 aa)" #@param ["Trp-cage (20 aa, tiny)", "Villin headpiece (36 aa)", "Protein G B1 domain (56 aa)", "Ubiquitin (76 aa)", "GFP (238 aa, slow on CPU)"]
import random, collections
import matplotlib.pyplot as plt
from esm.sdk.api import ESMProtein, GenerationConfig

seq = LIBRARY[sweep_protein]
fracs = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8]
ov, am, tp = [], [], []
for f in fracs:
    random.seed(0)
    idx = sorted(random.sample(range(len(seq)), int(len(seq) * f)))
    m = ''.join('_' if i in set(idx) else a for i, a in enumerate(seq))
    q = model.generate(ESMProtein(sequence=m),
                       GenerationConfig(track='sequence',
                                        num_steps=max(len(idx), 1), temperature=0.7))
    s = q.sequence
    ov.append(sum(a == b for a, b in zip(seq, s)) / len(seq))
    am.append(sum(seq[i] == s[i] for i in idx) / max(len(idx), 1))
    tp.append(collections.Counter(s).most_common(1)[0][1] / len(s))
    print(f'  {f:>4.0%} masked -> overall {ov[-1]:5.1%}   at masked {am[-1]:5.1%}   top residue {tp[-1]:5.1%}')

plt.figure(figsize=(7, 4.5))
plt.plot([f*100 for f in fracs], [v*100 for v in ov], 'o-', label='overall identity')
plt.plot([f*100 for f in fracs], [v*100 for v in am], 's-', label='identity at masked positions')
plt.plot([f*100 for f in fracs], [v*100 for v in tp], '^--', label='most common residue %')
plt.axhspan(0, 12, color='green', alpha=0.08)
plt.text(11, 6, 'normal composition', fontsize=8, color='green')
plt.xlabel('% of sequence masked'); plt.ylabel('%')
plt.title(sweep_protein); plt.legend(); plt.grid(alpha=0.3); plt.ylim(0, 100)
plt.show()


In [ ]:
#@title 💾 Download your design
from google.colab import files
files.download('design.pdb')


---
## Things to try, and one thing to look for

- **Run ubiquitin at 80% masking.** Look at the identity at masked positions.
  Then ask yourself what the model is actually doing. This is the most
  important result in the notebook.
- **Compare ubiquitin with Trp-cage** at the same masking fraction. Why might
  one be recalled far better than the other?
- **Drop the temperature to 0.3.** The output gets *more* confident and *less*
  protein-like. Why would that be?
- **Push to 90% masking.** At some point you are no longer redesigning a
  protein, you are asking the model to invent one. Watch what it does.
- **Paste in a protein you care about** and find its breaking point.

## For the discussion

Bring one number: the masking fraction at which *your* protein stopped being
itself, and how you decided that.

---

ESM3: Hayes et al., *Science* 2025. Model `esm3-sm-open-v1`, MIT licence.
